# คำนวณระยะทางถึงถนนจริง (OSM Road Centerline) — เวอร์ชันแก้ไข

**อัปเดตจากเวอร์ชันก่อนหน้า:** แก้ปัญหาความเข้ากันไม่ได้ของ `osmnx` เวอร์ชัน 2.x ซึ่งเปลี่ยนรูปแบบการส่งค่า bounding box (`bbox`) จากเดิม `(north, south, east, west)` เป็น `(west, south, east, north)` — สคริปต์นี้รองรับทั้งสองแบบอัตโนมัติ พร้อมเพิ่มเวลารอ (timeout) ให้นานขึ้นและแสดง error ที่อ่านง่ายขึ้นถ้ายังล้มเหลว

**อินพุต:** ไฟล์ `site_coordinates_resolved.csv` (ไฟล์พิกัดที่ดึงไว้แล้วจากรอบก่อนหน้า มีคอลัมน์ `site_id`, `lat`, `lon`)


## ขั้นตอนที่ 1: ติดตั้งไลบรารีที่จำเป็น

In [ ]:
!pip install osmnx geopandas shapely pyproj --quiet
import osmnx as ox
print("osmnx version:", ox.__version__)


In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from google.colab import files

ox.settings.timeout = 300          # เพิ่มเวลารอให้นานขึ้น (วินาที) เผื่อพื้นที่กว้าง
ox.settings.log_console = True     # แสดง log การเรียก Overpass API เพื่อช่วยดีบัก


## ขั้นตอนที่ 2: อัปโหลดไฟล์พิกัด 32 ไซต์

เลือกไฟล์ `site_coordinates_resolved.csv` ที่ได้จากรอบก่อนหน้า


In [ ]:
uploaded = files.upload()
coords_filename = list(uploaded.keys())[0]
sites = pd.read_csv(coords_filename)
sites = sites.dropna(subset=['lat', 'lon']).reset_index(drop=True)
print(f"จำนวนไซต์ที่มีพิกัดครบ: {len(sites)}")
sites.head()


## ขั้นตอนที่ 3: กำหนดขอบเขตพื้นที่และดึงโครงข่ายถนนจาก OSM

ฟังก์ชันด้านล่างลองเรียก `ox.graph_from_bbox` ด้วยรูปแบบ argument ทั้งสองแบบ (เวอร์ชันใหม่และเก่า) เพื่อความเข้ากันได้ ไม่ว่าเครื่องคุณจะลง osmnx เวอร์ชันไหน


In [ ]:
buffer_deg = 0.3
north = sites['lat'].max() + buffer_deg
south = sites['lat'].min() - buffer_deg
east = sites['lon'].max() + buffer_deg
west = sites['lon'].min() - buffer_deg

print(f"ขอบเขตพื้นที่: N={north:.4f}, S={south:.4f}, E={east:.4f}, W={west:.4f}")
print(f"ขนาดพื้นที่โดยประมาณ: {(north-south)*111:.0f} km (N-S) x {(east-west)*111:.0f} km (E-W)")

custom_filter = '["highway"~"motorway|trunk|primary|secondary"]'

def fetch_road_graph(north, south, east, west, custom_filter):
    errors = []

    # ลองแบบใหม่ก่อน: osmnx >= 2.0 ใช้ bbox=(west, south, east, north)
    try:
        print("กำลังลองรูปแบบ osmnx >= 2.0 : bbox=(west, south, east, north) ...")
        G = ox.graph_from_bbox(
            bbox=(west, south, east, north),
            custom_filter=custom_filter,
            simplify=True,
            retain_all=True
        )
        print("สำเร็จด้วยรูปแบบ osmnx >= 2.0")
        return G
    except Exception as e:
        errors.append(f"รูปแบบใหม่ (west,south,east,north) ล้มเหลว: {repr(e)}")

    # ลองแบบเก่า: osmnx < 2.0 ใช้ bbox=(north, south, east, west)
    try:
        print("กำลังลองรูปแบบ osmnx < 2.0 : bbox=(north, south, east, west) ...")
        G = ox.graph_from_bbox(
            bbox=(north, south, east, west),
            custom_filter=custom_filter,
            simplify=True,
            retain_all=True
        )
        print("สำเร็จด้วยรูปแบบ osmnx < 2.0")
        return G
    except Exception as e:
        errors.append(f"รูปแบบเก่า (north,south,east,west) ล้มเหลว: {repr(e)}")

    # ลองแบบ keyword เดี่ยว (บางเวอร์ชันกลาง ๆ รองรับ north=, south=, east=, west= แยกกัน)
    try:
        print("กำลังลองรูปแบบ north=/south=/east=/west= แยกพารามิเตอร์ ...")
        G = ox.graph_from_bbox(
            north=north, south=south, east=east, west=west,
            custom_filter=custom_filter,
            simplify=True,
            retain_all=True
        )
        print("สำเร็จด้วยรูปแบบพารามิเตอร์แยก")
        return G
    except Exception as e:
        errors.append(f"รูปแบบพารามิเตอร์แยก ล้มเหลว: {repr(e)}")

    raise RuntimeError(
        "ดึงโครงข่ายถนนไม่สำเร็จทั้ง 3 รูปแบบ:\n" + "\n".join(errors) +
        "\n\nลองตรวจสอบ: (1) การเชื่อมต่ออินเทอร์เน็ตของ Colab (2) เวอร์ชัน osmnx ด้วยคำสั่ง ox.__version__ "
        "(3) ลองรันใหม่อีกครั้งเผื่อ Overpass API ชั่วคราวล่ม (4) ลดขนาดพื้นที่ (buffer_deg) ถ้าพื้นที่ใหญ่เกินไป"
    )

G = fetch_road_graph(north, south, east, west, custom_filter)
print(f"\nจำนวนโหนด: {len(G.nodes)}, จำนวนเส้นทาง (edges): {len(G.edges)}")


## ขั้นตอนที่ 4: แปลงพิกัดเป็นระบบ UTM (EPSG:32647) เพื่อคำนวณระยะทางเป็นเมตรอย่างแม่นยำ


In [ ]:
G_proj = ox.project_graph(G, to_crs='EPSG:32647')

sites_gdf = gpd.GeoDataFrame(
    sites,
    geometry=[Point(xy) for xy in zip(sites['lon'], sites['lat'])],
    crs='EPSG:4326'
).to_crs('EPSG:32647')

X = sites_gdf.geometry.x.values
Y = sites_gdf.geometry.y.values
print("แปลงพิกัดเสร็จแล้ว")


## ขั้นตอนที่ 5: คำนวณระยะทางถึงถนนที่ใกล้ที่สุด (โครงข่ายทั้งหมด)


In [ ]:
nearest_edges, dists_m = ox.distance.nearest_edges(G_proj, X, Y, return_dist=True)
sites['dist_to_nearest_road_km'] = np.array(dists_m) / 1000
sites[['site_id', 'lat', 'lon', 'dist_to_nearest_road_km']].sort_values('dist_to_nearest_road_km')


## ขั้นตอนที่ 6: คำนวณระยะทางเฉพาะถึงถนนมิตรภาพ / ทางหลวงหมายเลข 2


In [ ]:
def is_mittraphap(data):
    name = data.get('name', '')
    ref = data.get('ref', '')
    names = name if isinstance(name, list) else [name]
    refs = ref if isinstance(ref, list) else [ref]
    names = [str(n) for n in names]
    refs = [str(r) for r in refs]
    name_match = any('mittraphap' in n.lower() or 'มิตรภาพ' in n for n in names)
    ref_match = any(r.strip() in ['2', 'AH1', 'AH 1'] for r in refs)
    return name_match or ref_match

mtr_edges = [(u, v, k) for u, v, k, data in G_proj.edges(keys=True, data=True) if is_mittraphap(data)]
print(f"จำนวนเส้นทางที่ตรงกับถนนมิตรภาพ/ทางหลวงหมายเลข 2: {len(mtr_edges)}")

if len(mtr_edges) == 0:
    print("⚠️ ไม่พบถนนที่ตรงเงื่อนไข ตัวอย่างชื่อ/รหัสถนนที่พบในพื้นที่ทั้งหมด (เพื่อช่วยดีบัก):")
    sample_tags = set()
    for u, v, k, data in list(G_proj.edges(keys=True, data=True))[:2000]:
        n = data.get('name'); r = data.get('ref')
        if n or r:
            sample_tags.add((str(n), str(r)))
    for t in list(sample_tags)[:30]:
        print("  ", t)
else:
    G_mtr = G_proj.edge_subgraph([(u, v, k) for u, v, k in mtr_edges]).copy()
    nearest_edges_mtr, dists_m_mtr = ox.distance.nearest_edges(G_mtr, X, Y, return_dist=True)
    sites['dist_to_mittraphap_road_km'] = np.array(dists_m_mtr) / 1000
    display(sites[['site_id', 'dist_to_nearest_road_km', 'dist_to_mittraphap_road_km']].sort_values('dist_to_mittraphap_road_km'))


## ขั้นตอนที่ 7: บันทึกผลลัพธ์และดาวน์โหลด


In [ ]:
output_filename = 'road_distance_osm_verified.csv'
sites.to_csv(output_filename, index=False, encoding='utf-8-sig')
files.download(output_filename)
print(f"บันทึกไฟล์: {output_filename}")


## ถ้ายังรันไม่ผ่าน ให้ทำอย่างไร

1. **คัดลอกข้อความ error เต็ม ๆ** จากเซลล์ที่รันไม่ผ่าน (โดยเฉพาะขั้นตอนที่ 3) แล้วส่งกลับมาในแชท จะช่วยวินิจฉัยได้ตรงจุดกว่าการเดา
2. ลองรันคำสั่งนี้ในเซลล์ใหม่เพื่อดูเวอร์ชันที่แน่นอน: `import osmnx; print(osmnx.__version__)`
3. ถ้า error เกี่ยวกับ Overpass API timeout หรือ "too many requests" ให้รอสัก 1-2 นาทีแล้วรันเซลล์ขั้นตอนที่ 3 ใหม่อีกครั้ง (บางครั้งเซิร์ฟเวอร์ Overpass สาธารณะมีคนใช้งานหนัก)
